# Tennis network analysis: full pipeline

Runs every computation of the study in order, from the raw files to the tables and figures of the report,
and shows the main results after each step.

**Before running**
1. From the repository root: `python -m venv .venv`, activate it, then `pip install -e ".[dev,notebook]"`.
2. Open this notebook with the `.venv` kernel (VS Code or `jupyter lab`).
3. Choose the settings in the next cell, then *Run All*.

**Running time.** With `QUICK = False` (the settings used in the report) the whole notebook takes about
one to two hours, most of it in the null models. `QUICK = True` uses few samples and finishes in minutes:
use it to check that everything works, **not** for the numbers of the report.

**Outputs.** Every step writes to `results/` and `report/`, overwriting the files committed in the
repository. `git diff --stat` shows what changed; `git checkout -- results report` restores the committed version.

In [ ]:
# Settings
QUICK = False  # True: few samples, for a test run only
DOWNLOAD = False  # True: download tennis-data.co.uk again (the files are committed)
SKIP = set()  # steps to skip, e.g. {"analysis"} if results/ is already up to date

In [ ]:
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, Markdown, display

from tennis_networks.config import REPORT, ROOT, TABLES, load_config

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

# Arguments of each step: full run (report) and quick run (test only).
ARGS = {
    "analysis": ([], ["--samples", "10"]),
    "hypotheses": (["--null", "200"], ["--null", "20", "--perm", "199"]),
    "structure": ([], ["--sw-samples", "3", "--perm", "199"]),
    "triads": ([], ["--null", "5", "--perm", "199"]),
    "levels": ([], ["--perm", "199"]),
}


def run(module, *extra):
    """Run one pipeline step as `python -m tennis_networks.<module>` and print its output."""
    if module in SKIP:
        print(f"skipped: {module}")
        return
    full, quick = ARGS.get(module, ([], []))
    cmd = [sys.executable, "-m", f"tennis_networks.{module}", *(quick if QUICK else full), *extra]
    start = time.time()
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-4000:])
        raise RuntimeError(f"{module} failed with exit code {proc.returncode}")
    print(f"[{module}] done in {(time.time() - start) / 60:.1f} min")


def table(name, **kwargs):
    return pd.read_csv(TABLES / name, **kwargs)


cfg = load_config()
print(f"Repository: {ROOT}")
print(f"Seed: {cfg['seed']}, quick mode: {QUICK}")

## 1. Data

Michieli (2018) networks from MATLAB to CSV edge lists, then the nine comparative networks from
tennis-data.co.uk (edges from loser to winner, weight = wins, walkovers excluded).

In [ ]:
if DOWNLOAD:
    run("download")
run("michieli")
run("tennisdata")
display(table("td_networks_summary.csv"))
display(table("td_validity_michieli.csv"))

## 2. Reproduction of Michieli (2018)

In [ ]:
run("reproduce")
display(table("michieli_reproduction.csv")[["network", "measure", "paper", "ours"]])

## 3. Measures of every network

Degrees, PageRank, betweenness, k-cores, components, distances, clustering, assortativity, triad census,
Louvain communities, robustness, and directed null models. The longest step.

In [ ]:
run("analysis")
cols = [
    "nodes",
    "edges",
    "density",
    "reciprocity",
    "giant_fraction",
    "avg_path_giant",
    "diameter_giant",
    "max_kcore",
    "modularity",
    "top10_pagerank_share",
    "gini_in_strength",
    "spearman_pagerank_atp_rank",
]
display(table("global_measures.csv", index_col="network")[cols].T)

## 4. Hypotheses H1-H4 (network level)

* H1: prestige more concentrated in 2007-2018 than in 2019-2025 (season permutation)
* H2: assortativity against the configuration model (with undirected clustering and transitivity)
* H3: grass more centralized than clay and hard (match permutation, size-aware)
* H4: WTA prestige less concentrated than ATP (paired seasons)

In [ ]:
run("hypotheses")
display(table("hypothesis_tests.csv"))
display(table("null_undirected.csv").pivot(index="network", columns="statistic", values="z"))

## 5. Structure: HITS, structural equivalence, core-periphery, small-world

* **HITS**: authority = beats players who reach the late rounds; hub = goes deep and loses to the champions.
* **Structural equivalence**: players with the same opponents; QAP against the gap in ranking.
* **Core-periphery** (Borgatti-Everett), compared with the k-core.
* **Small-world** sigma against the configuration model.

In [ ]:
run("structure")
display(table("structure_summary.csv", index_col="network").T)
hits = table("hits_top10.csv")
display(hits[hits["position"] <= 5].pivot(index=["network", "position"], columns="score", values="player"))
display(table("structural_equivalence_pairs.csv").groupby("network").head(3))

## 6. Triad census and H5

Ratio of cyclic triads (A beats B, B beats C, C beats A) to transitive ones: 1/3 with random results,
lower when the hierarchy is consistent. H5: the hierarchy was more rigid in 2007-2012.

In [ ]:
run("triads")
display(table("triads_networks.csv")[["network", "cyclic_ratio", "null_mean", "z", "p_empirical"]])
display(table("triads_h5.csv"))
table("triads_by_season.csv", index_col="season").plot(marker="o", title="Cyclic / transitive triads per season");

## 7. Hypotheses at the monadic, dyadic and mixed levels (slide L04)

* H6 (monadic): PageRank explains the ranking beyond the number of wins (permutation regression)
* H7 (dyadic): head-to-head dominance carries over between surfaces (QAP)
* H8 (mixed): players with the same surface profile meet more often (QAP)

In [ ]:
run("levels")
display(table("levels_tests.csv"))
profiles = table("surface_profiles.csv", index_col="player")
display(profiles["profile"].value_counts())

## 8. Report tables, figures and word count

In [ ]:
run("report_assets")
run("wordcount")
for name in ("fig_degree_ccdf", "fig_robustness", "fig_yearly"):
    display(Image(filename=str(REPORT / "figures" / f"{name}.png")))

## 9. Summary of all hypotheses

In [ ]:
h = table("hypothesis_tests.csv")
h5 = table("triads_h5.csv")
lv = table("levels_tests.csv")
summary = pd.concat(
    [
        h[["hypothesis", "statistic", "comparison", "value_a", "value_b", "p_value"]],
        h5[["hypothesis", "statistic", "comparison", "value_a", "value_b", "p_value"]],
        lv.rename(columns={"value": "value_a"})[["hypothesis", "statistic", "comparison", "value_a", "p_value"]],
    ],
    ignore_index=True,
)
display(summary)
display(Markdown(f"Results written to `{TABLES}` and `{REPORT}`. Check `git diff --stat` before committing."))